# 🥭 Train EfficientNet chấm chất lượng nông sản
**Làm theo từng bước, chạy từng cell một (Shift+Enter)**

Trước tiên: **Runtime → Change runtime type → T4 GPU → Save**

## Bước 1 — Kiểm tra GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU - kiểm tra lại Runtime!')
print('CUDA:', torch.cuda.is_available())

## Bước 2 — Cài thư viện

In [ ]:
!pip install roboflow pyyaml -q
print('Done!')

## Bước 3 — Tải dataset từ Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="aUahSiwi6a0Zfq1srrB9")
project = rf.workspace("bing-t3enk").project("fruits-quality-analysis")
version = project.version(1)
dataset = version.download("yolov11")

print('Dataset path:', dataset.location)

## Bước 4 — Xem dataset có những class nào

In [ ]:
import yaml, os
from pathlib import Path

DATASET_PATH = Path(dataset.location)

with open(DATASET_PATH / 'data.yaml') as f:
    cfg = yaml.safe_load(f)

CLASS_NAMES = cfg['names']
NUM_CLASSES = len(CLASS_NAMES)
print(f'Số class: {NUM_CLASSES}')
print(f'Tên class: {CLASS_NAMES}')

## Bước 5 — Convert YOLO format → Classification format

In [ ]:
import shutil
from collections import Counter

OUT_DIR = Path('/content/data')

total_copied = 0
skipped = 0

for split in ['train', 'valid', 'test']:
    out_split = 'val' if split in ['valid', 'test'] else 'train'
    img_dir = DATASET_PATH / split / 'images'
    lbl_dir = DATASET_PATH / split / 'labels'

    if not img_dir.exists():
        print(f'  Không tìm thấy {split}/, bỏ qua.')
        continue

    for img_path in list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            skipped += 1
            continue

        lines = lbl_path.read_text().strip().splitlines()
        if not lines:
            skipped += 1
            continue

        # Lấy class xuất hiện nhiều nhất trong ảnh
        ids = [int(l.split()[0]) for l in lines]
        cls_id = Counter(ids).most_common(1)[0][0]
        cls_name = CLASS_NAMES[cls_id]

        dst = OUT_DIR / out_split / cls_name
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copy(img_path, dst / img_path.name)
        total_copied += 1

print(f'\nĐã copy: {total_copied} ảnh | Bỏ qua: {skipped} ảnh')
print('\nPhân bố dataset:')
for split in ['train', 'val']:
    split_dir = OUT_DIR / split
    if split_dir.exists():
        for cls_dir in sorted(split_dir.iterdir()):
            n = len(list(cls_dir.glob('*')))
            print(f'  {split}/{cls_dir.name}: {n} ảnh')

## Bước 6 — Train EfficientNet

In [ ]:
import copy, time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

# ── Config ──
DATA_DIR   = '/content/data'
BATCH_SIZE = 32
NUM_EPOCHS = 30
LR         = 1e-3
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_PATH  = '/content/efficientnet_quality.pt'

print(f'Device: {DEVICE} | Classes: {NUM_CLASSES} | {CLASS_NAMES}')

# ── Transforms ──
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = datasets.ImageFolder(DATA_DIR + '/train', transform=train_tf)
val_ds   = datasets.ImageFolder(DATA_DIR + '/val',   transform=val_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

# ── Model ──
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
for p in model.parameters(): p.requires_grad = False
model.classifier[1] = nn.Linear(1280, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# ── Training ──
def run_epoch(loader, is_train):
    model.train() if is_train else model.eval()
    loss_sum, correct, total = 0, 0, 0
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labels)
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            loss_sum += loss.item() * imgs.size(0)
            correct  += (out.argmax(1) == labels).sum().item()
            total    += imgs.size(0)
    return loss_sum/total, correct/total

best_acc, best_w = 0, None
print('\nEpoch  | Train Loss | Train Acc | Val Loss | Val Acc')
print('-'*55)
for ep in range(1, NUM_EPOCHS+1):
    tl, ta = run_epoch(train_loader, True)
    vl, va = run_epoch(val_loader,   False)
    scheduler.step()
    flag = ' ← best' if va > best_acc else ''
    if va > best_acc:
        best_acc = va
        best_w   = copy.deepcopy(model.state_dict())
    print(f'  {ep:3d}  |   {tl:.4f}   |   {ta:.3f}   |  {vl:.4f}  |  {va:.3f}{flag}')

model.load_state_dict(best_w)
torch.save(model.state_dict(), SAVE_PATH)
print(f'\n✅ Saved → {SAVE_PATH}  (best val_acc={best_acc:.3f})')

## Bước 7 — Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().tolist()
        all_preds  += preds
        all_labels += labels.tolist()

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix (Val)')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()
print('Lớp nào hay bị nhầm → bổ sung thêm ảnh lớp đó!')

## Bước 8 — Tải weights về máy

In [ ]:
from google.colab import files
files.download('/content/efficientnet_quality.pt')
files.download('/content/confusion_matrix.png')
print('✅ File đã được tải xuống máy!')
print('Copy efficientnet_quality.pt vào D:/HocTap/Nam3/NongNghiepAI/Training/')